
```python
def test_get_idols_empty(client):
    response = client.get("/idols/")
    assert response.status_code == 200
    assert response.json()["items"] == []
    assert response.json()["total"] == 0
```

## Walking through the pieces, since this is your first one

**`def test_get_idols_empty(client):`** — `client` here is the fixture you built in `conftest.py`. Pytest sees your test function asks for a parameter named `client`, matches it to the fixture of the same name, and automatically runs that fixture first, handing you its `yield`ed value. This is pytest's core mechanic: **fixtures are requested by parameter name**, not imported or called directly.

**`client.get("/idols/")`** — this is `TestClient` (from `httpx`/Starlette) making a real HTTP-style request against your app, in-process — no actual server running, no real network call, but your route function genuinely executes, hits the (test) database, and returns a genuine response object.

**`assert response.status_code == 200`** — `assert` is Python's built-in test primitive: if the condition is `True`, nothing happens; if `False`, the test fails immediately with an `AssertionError`, and pytest reports exactly which assertion failed. This is the entire mechanism tests are built on — no special "test" syntax, just plain `assert` statements.

**Why check `items == []` and `total == 0`, specifically** — because your `db_session` fixture builds a completely empty test database for every test (recall `create_all`/`drop_all`). So this test is verifying: "when there's genuinely no data, does my pagination endpoint correctly report zero results, rather than erroring or returning something malformed?" This is deliberately the simplest possible case — confirming the baseline works before testing anything more complex.

Create the file, run `python -m pytest -v` (the `-v` flag shows each test by name, more informative than the bare summary you saw before), and paste the result.

Good that you did some reading before diving back in — let's map these to your actual project, not just definitions in the abstract.

## Function-based vs. class-based

**Function-based** — plain `def test_something():` — what you've written so far. Each test is fully independent, fixtures are requested as parameters.

**Class-based** — tests grouped inside a class (`class TestIdols:`), mainly for **organization**, not different mechanics:
```python
class TestIdols:
    def test_get_idols_empty(self, client):
        ...
    def test_create_idol(self, client):
        ...
```
The only real functional difference: a class can share setup across its methods more explicitly, and it groups related tests visually in output (`TestIdols::test_create_idol`).

**Recommendation for you: stick with function-based.** Class-based testing in pytest is largely a holdover from `unittest`-style testing (Python's older built-in framework) — pytest's actual idiomatic style, and what you'll see in most modern FastAPI codebases, is plain functions grouped by **file**, not by class (`test_idols.py`, `test_albums.py`, etc. — which you're already doing). Don't add class-based structure just because it exists; it doesn't buy you anything here.

## Fixtures — you already know this one

`conftest.py`'s `client`/`db_session` — reusable setup/teardown, requested by parameter name. You've used this correctly already.

## Marking (`@pytest.mark.X`)

Tags you attach to tests to control *how* they run. Most relevant ones for you:
- `@pytest.mark.skip(reason="...")` — skip a test temporarily (e.g. a feature not built yet).
- `@pytest.mark.slow` (custom marker) — tag expensive tests so you can run `pytest -m "not slow"` day-to-day and the full slow suite only occasionally.
- `@pytest.mark.parametrize` — covered next, technically a marker too.

For your project size right now: you likely won't need markers much beyond maybe tagging a couple of slower integration-style tests later. Not a priority.

## Parametrize — this one's genuinely useful for you, right now

Runs the *same* test body multiple times with different inputs, instead of copy-pasting near-identical test functions:
```python
import pytest

@pytest.mark.parametrize("bad_limit", [-1, 0, 101, 99999])
def test_get_idols_rejects_invalid_limit(client, bad_limit):
    response = client.get(f"/api/idols/?limit={bad_limit}")
    assert response.status_code == 422
```
This runs 4 separate test cases from one function — exactly the shape you'd want for testing your `Query(ge=1, le=100)` validation boundaries. **This one's worth using** — you have several routes with the same kind of boundary validation (skip/limit across 4 resources), and parametrize avoids writing near-duplicate test functions for each bad value.

## Mocking

Replacing a real dependency with a fake, controllable stand-in — typically for things that are slow, external, or non-deterministic (a real network call, a paid API, the current time). Example: if you build that `GET /idols/{id}/bio-summary` LLM endpoint you mentioned wanting to add later, you would **not** want your tests to make a real Anthropic API call every time (slow, costs money, and the response could vary) — you'd mock `anthropic.Client.messages.create` to return a canned fake response instead, and just verify your route handles that response correctly.

**Not needed for your project yet** — you have no external API calls in your current CRUD routes (everything talks to your own Postgres and Redis, both real, both fast, both fine to hit directly in tests). This becomes relevant the moment you build that LLM endpoint — good to know it exists, not urgent to learn deeply today.

## Bottom line for right now

Priority order for what actually matters to your project: **fixtures (done)** → **plain function-based tests** → **parametrize for the validation-boundary tests** → mocking and marking are "know they exist, use them when the actual need appears" (mocking specifically becomes relevant once you build the LLM endpoint).

Go ahead and write that create→read-back test now — plain function-based, no class, no marks needed yet.

Good — this is exactly what I needed. Let's build this in layers: first the fixtures for an authenticated admin client, then the actual idol tests, explaining each new piece as it appears.

## Step 1: extend `conftest.py` with an admin fixture

```python
from app.models import models
from app.config.enums import UserRole
from app.utils.security import hash_password  # adjust import to wherever hash_password actually lives


@pytest.fixture
def admin_user(db_session):
    user = models.User(
        username="admin_test",
        email="admin_test@example.com",
        hashed_password=hash_password("testpassword123"),
        role=UserRole.ADMIN,
    )
    db_session.add(user)
    db_session.commit()
    return user


@pytest.fixture
def admin_client(client, admin_user):
    response = client.post(
        "/api/users/login",
        data={"username": "admin_test", "password": "testpassword123"},
    )
    token = response.json()["access_token"]
    client.headers.update({"Authorization": f"Bearer {token}"})
    return client
```

## Explaining each new piece

**`admin_user(db_session)`** — this fixture *depends on* your existing `db_session` fixture (fixtures can request other fixtures, same as tests do). It inserts one real `User` row directly into the test database, with a real hashed password (using your actual `hash_password`, not a fake one — this matters, since `login` will call `verify_password` against it for real) and `role=UserRole.ADMIN`.

**`admin_client(client, admin_user)`** — this depends on *two* fixtures: the plain `client` you already built, and `admin_user` (which guarantees that admin exists in the DB *before* this fixture tries to log in as them — pytest resolves fixture dependencies in the right order automatically). It then does something important: it calls your **actual login route**, for real, through the same `TestClient` — not faking a token, genuinely exercising your login logic (username lookup, `verify_password`, token creation) exactly as production would. It grabs the real `access_token` from the response and attaches it as a default header on `client` going forward.

**Why `data={...}` and not `json={...}`** — your login route uses `OAuth2PasswordRequestForm`, which expects traditional form-encoded data (like an HTML form submission), not JSON. `httpx`'s `data=` parameter sends exactly that content type; `json=` would send the wrong format and your route wouldn't parse it correctly.

**Why mutate `client.headers` instead of returning a new object** — `TestClient.headers.update(...)` means every subsequent request made through this same `client` object automatically includes the `Authorization` header, without you having to pass `headers=...` manually on every single call in your tests. This is a deliberate convenience: any test that uses `admin_client` gets a pre-authenticated client for free.

## Step 2: the actual `Idol` tests — `test_idols.py`

```python
import pytest


def test_get_idols_empty(client):
    response = client.get("/api/idols/")
    assert response.status_code == 200
    assert response.json()["items"] == []
    assert response.json()["total"] == 0


def test_create_idol_requires_admin(client):
    response = client.post(
        "/api/idols/",
        json={"stage_name": "Somi", "gender": "F"},
    )
    assert response.status_code == 401


def test_create_idol_as_admin(admin_client, db_session):
    group = models.Group(name="Test Group", slug="test-group")
    db_session.add(group)
    db_session.commit()

    response = admin_client.post(
        "/api/idols/",
        json={"stage_name": "Somi", "gender": "F", "group_id": group.id},
    )
    assert response.status_code == 200
    data = response.json()
    assert data["stage_name"] == "Somi"
    assert data["id"] is not None


def test_create_duplicate_idol_conflict(admin_client, db_session):
    group = models.Group(name="Test Group", slug="test-group")
    db_session.add(group)
    db_session.commit()

    payload = {"stage_name": "Somi", "gender": "F", "group_id": group.id}
    first = admin_client.post("/api/idols/", json=payload)
    assert first.status_code == 200

    second = admin_client.post("/api/idols/", json=payload)
    assert second.status_code == 409


@pytest.mark.parametrize("bad_limit", [-1, 0, 101])
def test_get_idols_rejects_invalid_limit(client, bad_limit):
    response = client.get(f"/api/idols/?limit={bad_limit}")
    assert response.status_code == 422
```

## Explaining the new ideas here

**`test_create_idol_requires_admin` uses plain `client`, not `admin_client`** — deliberately. This test's whole point is confirming an *unauthenticated* request gets rejected. Using `client` (no `Authorization` header attached) is what makes this test meaningful — if you'd accidentally used `admin_client` here, you'd be testing the opposite of what the test name claims.

**Why `test_create_idol_as_admin` creates a `Group` first, manually, via `db_session`** — remember your own design rule: idols require a real, pre-existing group (`group_id` required in `IdolCreate`). Rather than going through `POST /api/groups/` (which would work too, but adds an extra HTTP round-trip and extra assertions to a test that isn't *about* group creation), it's common and accepted practice to set up prerequisite data directly through the ORM session when the test isn't specifically testing that creation path. This keeps the test focused on what it's actually verifying.

**The duplicate test reuses the exact same `payload` dict for both requests** — this is deliberate: it guarantees the second POST is a genuine, exact duplicate (same `stage_name` + same `group_id`), which is precisely what your `create_idol` route's uniqueness check is supposed to catch. If the second request returns `200` instead of `409`, that means your duplicate-check logic broke — this test exists specifically to catch that regression.

**`@pytest.mark.parametrize("bad_limit", [-1, 0, 101])`** — this is the parametrize case from earlier, now put to real use: three separate test runs, one per value, each confirming your `Query(ge=1, le=100)` validation genuinely rejects out-of-range limits with a `422`. One function, three test cases in your output.

Run `python -m pytest -v`, and paste the result — there's a good chance something here needs adjusting to your exact code (route paths, exact field names), so let's see what actually happens rather than assume it's perfect on the first try.